# Cross-Seed Latent Graph Consistency

Measures how consistently different training seeds discover the same latent relational structure.

**Workflow:**
1. Discover trial directories and select best checkpoints per seed.
2. Collect observations via agent rollout (one seed), keeping only steps where `max_rho > rho_thresh`.
3. Run all seed encoders on the shared observation set → posterior `[N_obs, E]` per seed.
4. Per observation step: compute pairwise Pearson r and top-K overlap across seeds → `[N_obs, S, S]`.
5. Average across steps → `S×S` heatmaps.

In [1]:
%matplotlib inline
import logging
import sys
from pathlib import Path
import os

# Import matplotlib.pyplot BEFORE cross_seed_analysis so that its
# module-level `matplotlib.use("Agg")` call becomes a no-op (backend
# cannot be switched after pyplot is already active).
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import pearsonr

logging.basicConfig(level=logging.WARNING)
plt.style.use("default")

# Project root must be on PYTHONPATH (e.g. run from repo root or set below)
REPO_ROOT = Path("/home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids").resolve()
os.chdir(REPO_ROOT)
SRC = REPO_ROOT / "src"
EXPERIMENTS = REPO_ROOT / "experiments"
for p in [str(SRC), str(EXPERIMENTS), str(REPO_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Reuse building blocks from the existing analysis script
from analysis.cross_seed_analysis import (
    find_trial_dirs,
    get_seed,
    best_checkpoint,
    load_agent,
    compute_posteriors,
    plot_matrix_heatmap,
)

## Configuration

In [2]:
# ── experiment to analyse ──────────────────────────────────────────────────────
EXPERIMENT_DIR = REPO_ROOT / "results/2026_07_20_IEEE14/rappo_multiseed_gcnconv"
OUT_DIR        = EXPERIMENT_DIR / "cross_seed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── observation collection ─────────────────────────────────────────────────────
ENV_NAME   = "l2rpn_case14_sandbox_test"  # Grid2Op env used for inference
N_OBS      = 100   # total observations to collect
RHO_THRESH = 0.95  # min rho_max to include an observation

# ── analysis ───────────────────────────────────────────────────────────────────
# K is set automatically to n_line after the env is loaded, but can be overridden
K_OVERRIDE  = 40   # e.g. 20; None → use n_line
CONV_TYPE   = None   # None → from checkpoint config; "gin" for pre-revert ckpts
CHECKPOINT_NAME = None  # None → best per trial from progress.csv

## Ray Init & Trial Discovery

In [3]:
import ray

ray.init(
    ignore_reinit_error=True,
    logging_level=logging.ERROR,
    log_to_driver=False,
    num_cpus=1,
    object_store_memory=512 * 1024 * 1024,
)

trial_dirs = find_trial_dirs(EXPERIMENT_DIR)
print(f"Found {len(trial_dirs)} trial dir(s) in {EXPERIMENT_DIR}")

# Map seed → (trial_dir, checkpoint_name)
seed_to_info: dict[int, tuple[Path, str]] = {}
for td in trial_dirs:
    try:
        seed = get_seed(td)
    except KeyError as e:
        print(f"  WARNING: skipping {td.name} — {e}")
        continue
    try:
        ckpt = CHECKPOINT_NAME or best_checkpoint(td)
    except FileNotFoundError as e:
        print(f"  WARNING: skipping {td.name} — {e}")
        continue
    seed_to_info[seed] = (td, ckpt)
    print(f"  seed={seed}  checkpoint={ckpt}  trial={td.name}")

seeds = sorted(seed_to_info.keys())
print(f"\nSeeds: {seeds}")

/home/adrian/.conda/envs/L2RPN/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-07-23 13:24:11,913	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Found 5 trial dir(s) in /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv
  seed=0  checkpoint=checkpoint_000008  trial=CustomPPO_RARL_5887369_1b111_2026-07-18_21-30-16
  seed=1  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887370_1ae2e_2026-07-18_21-30-16
  seed=2  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887371_15dd9_2026-07-18_21-30-08
  seed=3  checkpoint=checkpoint_000007  trial=CustomPPO_RARL_5887372_1a964_2026-07-18_21-30-16
  seed=4  checkpoint=checkpoint_000006  trial=CustomPPO_RARL_5887373_1c693_2026-07-18_21-30-19

Seeds: [0, 1, 2, 3, 4]


## Observation Collection via Agent Rollout

One seed's trained agent acts in the environment. Observations where `max_rho > RHO_THRESH` are added to the shared set used for all seeds.

In [4]:
def collect_obs_via_rollout(
    agent,
    g2op_env,
    n_target: int,
    rho_thresh: float = 0.95,
    max_chronics: int = 50,
    seed: int = 42,
) -> list:
    """
    Collect Grid2Op observations by running an agent rollout.

    Iterates over chronics in random order. Within each episode the agent acts
    normally; every time step where ``obs.rho.max() > rho_thresh`` the observation
    is added to the set. Stops when ``n_target`` observations are collected or
    ``max_chronics`` episodes have been attempted.

    :param agent: Loaded RllibAgent whose ``act()`` drives the rollout.
    :param g2op_env: Underlying Grid2Op environment.
    :param n_target: Desired number of observations.
    :param rho_thresh: Minimum ``rho.max()`` for an observation to be included.
    :param max_chronics: Upper bound on the number of episodes to try.
    :param seed: RNG seed for chronic shuffling.
    :return: List of copied ``BaseObservation`` objects.
    """
    n_chronics = len(g2op_env.chronics_handler.subpaths)
    chronic_ids = list(range(n_chronics))
    np.random.default_rng(seed).shuffle(chronic_ids)

    collected = []
    for chronic_id in chronic_ids[:max_chronics]:
        if len(collected) >= n_target:
            break

        g2op_env.set_id(chronic_id)
        obs = g2op_env.reset()
        reward, done = 0.0, False

        while not done:
            action = agent.activation_function(obs, reward, done)
            obs, reward, done, _ = g2op_env.step(action)

            if float(obs.rho.max()) > rho_thresh:
                collected.append(obs.copy())
                if len(collected) >= n_target:
                    break

    if not collected:
        raise RuntimeError(
            f"No observations collected with rho > {rho_thresh}. "
            "Try lowering RHO_THRESH or increasing max_chronics."
        )

    rhos = [float(o.rho.max()) for o in collected]
    print(
        f"Collected {len(collected)} obs from up to {min(max_chronics, n_chronics)} chronics  "
        f"rho=[{min(rhos):.2f}, {max(rhos):.2f}]  mean={np.mean(rhos):.2f}"
    )
    return collected

In [5]:
# Use the first seed's agent for the rollout; keep wrappers alive throughout analysis
rollout_seed = seeds[0]
rollout_trial, rollout_ckpt = seed_to_info[rollout_seed]
print(f"Loading seed {rollout_seed} for rollout (checkpoint: {rollout_ckpt}) ...")

rollout_agent, rollout_g2op_env, rollout_gym_wrapper = load_agent(
    rollout_trial, rollout_ckpt, ENV_NAME, conv_type=CONV_TYPE
)

obs_list = collect_obs_via_rollout(
    rollout_agent, rollout_g2op_env,
    n_target=N_OBS, rho_thresh=RHO_THRESH,
)
print(f"Observation set size: {len(obs_list)}")

Loading seed 0 for rollout (checkpoint: checkpoint_000008) ...


/home/adrian/.conda/envs/L2RPN/lib/python3.10/site-packages/grid2op/MakeEnv/Make.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Collected 100 obs from up to 50 chronics  rho=[0.95, 0.99]  mean=0.96
Observation set size: 100


## Per-Seed Inference

Run each seed's encoder on the shared observation set → `posteriors[seed]: [N_obs, E]`.

In [6]:
seed_posteriors: dict[int, np.ndarray] = {}
gym_wrappers = [rollout_gym_wrapper]  # keep all wrappers alive

for s in seeds:
    trial_dir, ckpt_name = seed_to_info[s]
    print(f"Seed {s}: loading {ckpt_name} from {trial_dir.name} ...")
    agent, g2op_env, gym_wrapper = load_agent(trial_dir, ckpt_name, ENV_NAME, conv_type=CONV_TYPE)
    gym_wrappers.append(gym_wrapper)

    probs = compute_posteriors(agent, obs_list)   # [N_obs, E]
    seed_posteriors[s] = probs
    print(f"  posteriors: shape={probs.shape}  mean={probs.mean():.4f}  std={probs.std():.4f}")

N_obs = len(obs_list)
E     = next(iter(seed_posteriors.values())).shape[1]
S     = len(seeds)
print(f"\nDone. N_obs={N_obs}  E={E}  S={S}")

Seed 0: loading checkpoint_000008 from CustomPPO_RARL_5887369_1b111_2026-07-18_21-30-16 ...
  posteriors: shape=(100, 3192)  mean=0.0363  std=0.0451
Seed 1: loading checkpoint_000007 from CustomPPO_RARL_5887370_1ae2e_2026-07-18_21-30-16 ...
  posteriors: shape=(100, 3192)  mean=0.0146  std=0.0456
Seed 2: loading checkpoint_000007 from CustomPPO_RARL_5887371_15dd9_2026-07-18_21-30-08 ...
  posteriors: shape=(100, 3192)  mean=0.0864  std=0.1437
Seed 3: loading checkpoint_000007 from CustomPPO_RARL_5887372_1a964_2026-07-18_21-30-16 ...
  posteriors: shape=(100, 3192)  mean=0.0185  std=0.0827
Seed 4: loading checkpoint_000006 from CustomPPO_RARL_5887373_1c693_2026-07-18_21-30-19 ...
  posteriors: shape=(100, 3192)  mean=0.0493  std=0.0867

Done. N_obs=100  E=3192  S=5


## Per-Step Metrics

For each observation step $t$ compute the $S \times S$ pairwise metric matrix, then average across all steps.

In [7]:
def pearson_per_step(
    seed_posteriors: dict[int, np.ndarray],
) -> tuple[np.ndarray, np.ndarray, list[int]]:
    """
    Compute pairwise Pearson r between seed posteriors at each observation step.

    :param seed_posteriors: ``{seed: [N_obs, E]}`` interaction probability arrays.
    :return:
        - ``mean_r`` – ``[S, S]`` mean Pearson r across all steps.
        - ``step_r`` – ``[N_obs, S, S]`` per-step matrices.
        - ``seeds``  – sorted seed list (row/column order).
    """
    sorted_seeds = sorted(seed_posteriors.keys())
    stacked = np.stack([seed_posteriors[s] for s in sorted_seeds])  # [S, N_obs, E]
    S, N, _ = stacked.shape

    step_r = np.ones((N, S, S), dtype=np.float32)
    for t in range(N):
        for i in range(S):
            for j in range(i + 1, S):
                r, _ = pearsonr(stacked[i, t], stacked[j, t])
                step_r[t, i, j] = step_r[t, j, i] = float(r)

    return step_r.mean(axis=0), step_r, sorted_seeds


def top_k_overlap_per_step(
    seed_posteriors: dict[int, np.ndarray],
    k: int,
) -> tuple[np.ndarray, np.ndarray, list[int]]:
    """
    Compute pairwise top-K edge overlap between seeds at each observation step.

    :param seed_posteriors: ``{seed: [N_obs, E]}`` interaction probability arrays.
    :param k: Number of top edges to compare.
    :return:
        - ``mean_overlap`` – ``[S, S]`` mean top-K overlap across all steps.
        - ``step_overlap`` – ``[N_obs, S, S]`` per-step matrices.
        - ``seeds``        – sorted seed list (row/column order).
    """
    sorted_seeds = sorted(seed_posteriors.keys())
    stacked = np.stack([seed_posteriors[s] for s in sorted_seeds])  # [S, N_obs, E]
    S, N, _ = stacked.shape

    step_overlap = np.ones((N, S, S), dtype=np.float32)
    for t in range(N):
        top_k_sets = [
            set(np.argsort(stacked[i, t])[-k:]) for i in range(S)
        ]
        for i in range(S):
            for j in range(i + 1, S):
                overlap = len(top_k_sets[i] & top_k_sets[j]) / k
                step_overlap[t, i, j] = step_overlap[t, j, i] = float(overlap)

    return step_overlap.mean(axis=0), step_overlap, sorted_seeds

## Pearson r Heatmap

In [8]:
mean_r, step_r, _ = pearson_per_step(seed_posteriors)

off_diag = mean_r[np.triu_indices(S, k=1)]
print(f"Mean Pearson r  — mean={off_diag.mean():.3f}  min={off_diag.min():.3f}  max={off_diag.max():.3f}")

plot_matrix_heatmap(
    mean_r, seeds,
    out_path=OUT_DIR / "pearson_heatmap_perstep.png",
    title="Cross-seed Pearson r (mean over per-step pairwise correlations)",
    cbar_label="Pearson r",
    vmin=-1.0,
    vmax=1.0,
)
plt.imshow(mean_r, vmin=-1, vmax=1, cmap="RdYlGn")
plt.colorbar(label="Pearson r")
plt.xticks(range(S), [f"seed {s}" for s in seeds], rotation=45, ha="right")
plt.yticks(range(S), [f"seed {s}" for s in seeds])
plt.title(f"Cross-seed Pearson r\n(mean over per-step pairwise correlations)")
plt.tight_layout()
plt.show()

Mean Pearson r  — mean=0.051  min=-0.048  max=0.186
Saved: /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv/cross_seed/pearson_heatmap_perstep.png


/tmp/ipykernel_27322/4254499320.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Top-K Overlap Heatmap

In [9]:
n_line  = rollout_g2op_env.n_line
K = K_OVERRIDE if K_OVERRIDE is not None else n_line
mean_overlap, step_overlap, _ = top_k_overlap_per_step(seed_posteriors, k=K)

off_diag_k = mean_overlap[np.triu_indices(S, k=1)]
print(f"Mean top-{K} overlap — mean={off_diag_k.mean():.3f}  min={off_diag_k.min():.3f}  max={off_diag_k.max():.3f}")

plot_matrix_heatmap(
    mean_overlap, seeds,
    out_path=OUT_DIR / "topk_overlap_heatmap_perstep.png",
    title=f"Cross-seed top-{K} edge overlap (mean over per-step pairwise overlap)",
    cbar_label=f"Top-{K} overlap",
    vmin=0.0,
    vmax=1.0,
    fmt=".2f",
)
plt.imshow(mean_overlap, vmin=0, vmax=1, cmap="RdYlGn")
plt.colorbar(label=f"Top-{K} overlap")
plt.xticks(range(S), [f"seed {s}" for s in seeds], rotation=45, ha="right")
plt.yticks(range(S), [f"seed {s}" for s in seeds])
plt.title(f"Cross-seed top-{K} overlap\nmean={off_diag_k.mean():.3f}")
plt.tight_layout()
plt.show()

Mean top-40 overlap — mean=0.023  min=0.001  max=0.093
Saved: /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv/cross_seed/topk_overlap_heatmap_perstep.png


/tmp/ipykernel_27322/3205585169.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Save Raw Data

In [10]:
npz_path = OUT_DIR / "posteriors_perstep.npz"
np.savez(
    npz_path,
    seeds=np.array(seeds),
    mean_pearson=mean_r,
    step_pearson=step_r,
    mean_topk=mean_overlap,
    step_topk=step_overlap,
    **{f"posteriors_seed{s}": seed_posteriors[s] for s in seeds},
)
print(f"Saved: {npz_path}")

Saved: /home/adrian/Dev/NRI-for-explainable-RL-in-Power-Grids/results/2026_07_20_IEEE14/rappo_multiseed_gcnconv/cross_seed/posteriors_perstep.npz


## Cleanup

In [11]:
ray.shutdown()